# Hợp nhất Mô hình (Model Merge: Base Model + 1-LoRA Adapter)

Notebook này thực hiện việc **hợp nhất (merge) adapter LoRA** đã được huấn luyện ở Notebook 8 quay trở lại **mô hình nền Llama-3.1-8B gốc**.

**Cấu hình Cache & Tránh rác ổ C:**
- Sử dụng tệp `.env` để cấu hình đường dẫn cache Hugging Face và Unsloth trên ổ `T:`. Toàn bộ dữ liệu mô hình tải về sẽ không lưu ở ổ C.

**Tại sao cần Model Merge?**
- Sau khi hợp nhất, chúng ta có một mô hình độc lập (single weights file) có thể được chạy trực tiếp qua HuggingFace, vLLM, hoặc chuyển đổi sang định dạng GGUF để chạy cục bộ bằng Ollama/Llama.cpp.
- Hợp nhất giúp tối ưu hóa đáng kể tốc độ suy luận (Inference Latency) so với việc nạp song song mô hình nền và adapter LoRA.

In [ ]:
import os
from dotenv import load_dotenv

# Nạp các biến môi trường cấu hình cache
load_dotenv(os.path.abspath("../.env"))

from unsloth import FastLanguageModel

## 1. Thiết lập Cấu hình & Tải Adapter và Mô hình nền

In [ ]:
max_seq_length = 2048
dtype = None
load_in_4bit = True # Giữ nguyên chế độ 4-bit để tiết kiệm RAM trên laptop

ADAPTER_DIR = "../adapters/llama_8b_1lora_aes"

print(f"Đang tải mô hình nền và adapter LoRA từ: {ADAPTER_DIR}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = ADAPTER_DIR,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

## 2. Thực hiện Hợp nhất và Lưu trữ Cục bộ

Unsloth hỗ trợ 2 chế độ hợp nhất chính:
1. **`merged_16bit`**: Xuất ra mô hình đầy đủ độ chính xác float16 (~16 GB). Đây là phương án khuyến nghị nếu bạn muốn export sang GGUF hoặc vLLM để triển khai ở môi trường sản xuất.
2. **`merged_4bit`**: Xuất ra mô hình đã được lượng hóa 4-bit sẵn (~5.5 GB). Tiết kiệm ổ cứng tối đa.

In [ ]:
MERGED_MODEL_DIR_16BIT = "../merged_model_16bit"
MERGED_MODEL_DIR_4BIT = "../merged_model_4bit"

# 1. Xuất mô hình Merged 16-bit (Có thể mất từ 5-10 phút để lưu trữ)
print(f"Đang hợp nhất và lưu mô hình Float16 tại: {MERGED_MODEL_DIR_16BIT}...")
model.save_pretrained_merged(
    MERGED_MODEL_DIR_16BIT, 
    tokenizer, 
    save_method = "merged_16bit"
)
print("✔ Hoàn thành lưu mô hình 16-bit!")

# 2. Xuất mô hình Merged 4-bit (Nhẹ hơn, tiết kiệm ổ cứng)
print(f"\nĐang hợp nhất và lưu mô hình lượng hóa 4-bit tại: {MERGED_MODEL_DIR_4BIT}...")
model.save_pretrained_merged(
    MERGED_MODEL_DIR_4BIT, 
    tokenizer, 
    save_method = "merged_4bit"
)
print("✔ Hoàn thành lưu mô hình 4-bit!")

## 3. Hướng dẫn Đẩy mô hình lên Hugging Face Hub (Tùy chọn)

Nếu muốn sao lưu hoặc chia sẻ với cộng đồng, bạn có thể đẩy thẳng mô hình đã merge lên Hugging Face Hub:

In [ ]:
# # Đăng nhập Hugging Face bằng token ghi (write token)
# # huggingface-cli login

# # Lệnh đẩy lên Hub:
# model.push_to_hub_merged(
#     "username/llama-3.1-8b-aes-1lora", 
#     tokenizer, 
#     save_method = "merged_16bit", 
#     token = "your_hf_token"
# )